## 01. Setup Dependencies

In [ ]:
import sys, subprocess, importlib.util, os

PINNED_PACKAGES = [
    "transformers==4.46.3",
    "accelerate==0.33.0",
    "sentencepiece>=0.2.0",
    "safetensors>=0.4.5",
    "scikit-learn>=1.5.2,<1.8",
    "lightgbm>=4.3.0",
    "optuna>=3.6.1",
    "openpyxl>=3.1.2",
    "tqdm>=4.66.1"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + PINNED_PACKAGES)
print("Dependency setup selesai.")


Dependency setup selesai.


## 02. Imports and Global Seed

In [ ]:
import os, re, gc, json, math, random, warnings, shutil, html, time
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.special import softmax

from IPython.display import display
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, recall_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn import set_config

warnings.filterwarnings("ignore")

try:
    set_config(enable_metadata_routing=False)
except Exception:
    pass

try:
    import torch
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForMaskedLM, DataCollatorForLanguageModeling, get_cosine_schedule_with_warmup
    TRANSFORMERS_AVAILABLE = True
except Exception as e:
    print("Transformers/PyTorch tidak tersedia:", repr(e))
    TRANSFORMERS_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception as e:
    print("LightGBM tidak tersedia:", repr(e))
    LGBM_AVAILABLE = False

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if TRANSFORMERS_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)
DEVICE = "cuda" if TRANSFORMERS_AVAILABLE and torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 03. Configuration

In [ ]:
TEAM_NAME = "TELULANG"
AUTO_UPLOAD_IF_MISSING = True
USE_GOOGLE_DRIVE_CHECKPOINTS = False

RUN_TFIDF = True
RUN_TFIDF_NOISE_RETRAIN = True
RUN_BINARY_GATES = True
RUN_PAIRWISE = True
RUN_TAPT = True
RUN_TRANSFORMERS = True
RUN_META_STACKING = True
RUN_LGBM_META = True
RUN_PER_CLASS_WEIGHTING = True
RUN_POSTPROCESSING = True
MAKE_DIVERSE_SUBMISSIONS = True

N_SPLITS_TFIDF = 10
N_SPLITS_BINARY = 5
N_SPLITS_PAIRWISE = 5
N_SPLITS_TRANSFORMER = 5
N_SPLITS_META = 5

USE_AMP = True
USE_BF16 = True
USE_GRADIENT_CHECKPOINTING = True
USE_LABEL_SMOOTHING = True
LABEL_SMOOTHING = 0.04
USE_TTA = True
ADD_SPECIAL_KEYWORD_TOKENS = True
NUM_WORKERS = 2
PIN_MEMORY = True
PERSISTENT_WORKERS = False
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.08
WEIGHT_DECAY = 0.01

TAPT_EPOCHS = 2
TAPT_BATCH_SIZE = 32
TAPT_MAX_LEN = 192
TAPT_LR = 5e-5
TAPT_MLM_PROB = 0.15
TAPT_USE_TEST_TEXT = True

if TRANSFORMERS_AVAILABLE and DEVICE == "cuda":
    USE_BF16 = bool(USE_BF16 and torch.cuda.is_bf16_supported())
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
else:
    USE_AMP = False
    USE_BF16 = False
    AMP_DTYPE = None

LABELS = ["Anggaran", "Distribusi", "Ekonomi", "Kualitas Pangan", "Lainnya", "Politik", "Sasaran Penerima", "Tata Kelola"]
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}
NUM_LABELS = len(LABELS)

SPECIAL_TOKENS = ["[KW_" + lab.upper().replace(" ", "_") + "]" for lab in LABELS]

RECALL_FLOORS = {
    "Anggaran": 0.62,
    "Distribusi": 0.64,
    "Ekonomi": 0.68,
    "Kualitas Pangan": 0.66,
    "Lainnya": 0.62,
    "Politik": 0.64,
    "Sasaran Penerima": 0.64,
    "Tata Kelola": 0.64,
}

TRANSFORMER_MODEL_CONFIGS = [
    {"alias": "tapt_indobert_large_nat_bs_s42", "model_name": "indobenchmark/indobert-large-p1", "seed": 42, "use_hint": False, "use_tapt": True, "loss_mode": "balanced_softmax", "use_sampler": False, "use_noise_weight": True, "epochs": 5, "max_len": 192, "batch_size": 16, "eval_batch_size": 48, "lr": 1.20e-5, "head_lr": 4.0e-5},
    {"alias": "tapt_indobert_large_hint_cw_s343", "model_name": "indobenchmark/indobert-large-p1", "seed": 343, "use_hint": True, "use_tapt": True, "loss_mode": "class_weight", "use_sampler": True, "use_noise_weight": True, "epochs": 5, "max_len": 192, "batch_size": 16, "eval_batch_size": 48, "lr": 1.20e-5, "head_lr": 4.0e-5},
    {"alias": "indobert_large_nat_ce_s2026", "model_name": "indobenchmark/indobert-large-p1", "seed": 2026, "use_hint": False, "use_tapt": False, "loss_mode": "ce", "use_sampler": True, "use_noise_weight": False, "epochs": 4, "max_len": 192, "batch_size": 16, "eval_batch_size": 48, "lr": 1.10e-5, "head_lr": 3.8e-5},
    {"alias": "tapt_indobert_base_p1_nat_bs_s777", "model_name": "indobenchmark/indobert-base-p1", "seed": 777, "use_hint": False, "use_tapt": True, "loss_mode": "balanced_softmax", "use_sampler": False, "use_noise_weight": True, "epochs": 5, "max_len": 224, "batch_size": 32, "eval_batch_size": 96, "lr": 1.70e-5, "head_lr": 5.0e-5},
    {"alias": "tapt_indobert_base_p1_hint_sampler_s934", "model_name": "indobenchmark/indobert-base-p1", "seed": 934, "use_hint": True, "use_tapt": True, "loss_mode": "ce", "use_sampler": True, "use_noise_weight": True, "epochs": 5, "max_len": 224, "batch_size": 32, "eval_batch_size": 96, "lr": 1.70e-5, "head_lr": 5.0e-5},
    {"alias": "tapt_indobert_base_p2_hint_cw_s2025", "model_name": "indobenchmark/indobert-base-p2", "seed": 2025, "use_hint": True, "use_tapt": True, "loss_mode": "class_weight", "use_sampler": False, "use_noise_weight": True, "epochs": 5, "max_len": 224, "batch_size": 32, "eval_batch_size": 96, "lr": 1.70e-5, "head_lr": 5.0e-5},
    {"alias": "cahya_bert_nat_bs_s52", "model_name": "cahya/bert-base-indonesian-522M", "seed": 52, "use_hint": False, "use_tapt": False, "loss_mode": "balanced_softmax", "use_sampler": False, "use_noise_weight": True, "epochs": 4, "max_len": 192, "batch_size": 16, "eval_batch_size": 48, "lr": 1.35e-5, "head_lr": 4.0e-5},
    {"alias": "cahya_bert_hint_cw_s934", "model_name": "cahya/bert-base-indonesian-522M", "seed": 934, "use_hint": True, "use_tapt": False, "loss_mode": "class_weight", "use_sampler": True, "use_noise_weight": True, "epochs": 4, "max_len": 192, "batch_size": 16, "eval_batch_size": 48, "lr": 1.35e-5, "head_lr": 4.0e-5},
    {"alias": "xlmr_large_nat_bs_s52", "model_name": "xlm-roberta-large", "seed": 52, "use_hint": False, "use_tapt": False, "loss_mode": "balanced_softmax", "use_sampler": False, "use_noise_weight": True, "epochs": 4, "max_len": 192, "batch_size": 12, "eval_batch_size": 32, "lr": 1.0e-5, "head_lr": 3.0e-5},
    {"alias": "xlmr_large_hint_cw_s123", "model_name": "xlm-roberta-large", "seed": 123, "use_hint": True, "use_tapt": False, "loss_mode": "class_weight", "use_sampler": True, "use_noise_weight": True, "epochs": 4, "max_len": 192, "batch_size": 12, "eval_batch_size": 32, "lr": 1.0e-5, "head_lr": 3.0e-5},
]

CACHE_VERSION = "mbg_a100_max_clean_v4_tapt_lossdiverse"

def setup_dirs():
    base = Path("mbg_bdc_a100_work")
    if USE_GOOGLE_DRIVE_CHECKPOINTS:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            base = Path("/content/drive/MyDrive/MBG_BDC_A100_MAX")
        except Exception as e:
            print("Drive checkpoint tidak aktif:", repr(e))
            base = Path("mbg_bdc_a100_work")
    out = base / "outputs"
    ckpt = base / "checkpoints"
    cache = base / "cache"
    out.mkdir(parents=True, exist_ok=True)
    ckpt.mkdir(parents=True, exist_ok=True)
    cache.mkdir(parents=True, exist_ok=True)
    return base, out, ckpt, cache

WORK_DIR, OUTPUT_DIR, CHECKPOINT_DIR, CACHE_DIR = setup_dirs()
print("Configuration loaded.")
print("Output dir:", OUTPUT_DIR)
print("Transformer models:", [cfg["alias"] for cfg in TRANSFORMER_MODEL_CONFIGS])
print("AMP/BF16:", USE_AMP, USE_BF16, AMP_DTYPE)
print("TAPT:", RUN_TAPT, "Special tokens:", ADD_SPECIAL_KEYWORD_TOKENS)


## 04. Load and Validate Data

In [ ]:
def upload_files_colab():
    try:
        from google.colab import files
        print("Upload file case_1_labeled_data.xlsx, case_1_text_to_predict.xlsx, dan case_1_template_sheet.xlsx")
        uploaded = files.upload()
        print("Uploaded:", list(uploaded.keys()))
    except Exception as e:
        print("Upload Colab tidak tersedia:", repr(e))


def resolve_file(canonical_name):
    p = Path(canonical_name)
    if p.exists():
        return p
    stem, suffix = p.stem, p.suffix
    roots = [Path("."), Path("/content"), Path("/mnt/data")]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(sorted(root.glob(f"{stem}*{suffix}")))
    candidates = [c for c in candidates if c.is_file()]
    if candidates:
        chosen = sorted(candidates, key=lambda x: (len(x.name), str(x)))[0]
        print(f"Resolved {canonical_name} -> {chosen}")
        return chosen
    raise FileNotFoundError(canonical_name)


def load_all_files():
    try:
        return resolve_file("case_1_labeled_data.xlsx"), resolve_file("case_1_text_to_predict.xlsx"), resolve_file("case_1_template_sheet.xlsx")
    except FileNotFoundError:
        if AUTO_UPLOAD_IF_MISSING:
            upload_files_colab()
            return resolve_file("case_1_labeled_data.xlsx"), resolve_file("case_1_text_to_predict.xlsx"), resolve_file("case_1_template_sheet.xlsx")
        raise

train_path, test_path, template_path = load_all_files()
train_df_raw = pd.read_excel(train_path)
test_df_raw = pd.read_excel(test_path)
template_df = pd.read_excel(template_path)

TEXT_COL = "full_text" if "full_text" in train_df_raw.columns else train_df_raw.columns[0]
LABEL_COL = "label" if "label" in train_df_raw.columns else train_df_raw.columns[-1]
TEST_TEXT_COL = "full_text" if "full_text" in test_df_raw.columns else test_df_raw.columns[-1]
ID_COL = "id" if "id" in test_df_raw.columns else test_df_raw.columns[0]

train_df = train_df_raw[[TEXT_COL, LABEL_COL]].copy()
test_df = test_df_raw[[ID_COL, TEST_TEXT_COL]].copy()
train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").astype(str)
test_df[TEST_TEXT_COL] = test_df[TEST_TEXT_COL].fillna("").astype(str)
train_df[LABEL_COL] = train_df[LABEL_COL].astype(str).str.strip()

unknown_labels = sorted(set(train_df[LABEL_COL]) - set(LABELS))
if unknown_labels:
    raise ValueError(f"Label tidak dikenal: {unknown_labels}")
if len(test_df) != len(template_df):
    raise ValueError("Jumlah baris test dan template tidak sama.")
if ID_COL in template_df.columns:
    if test_df[ID_COL].astype(str).tolist() != template_df[ID_COL].astype(str).tolist():
        raise ValueError("Urutan ID test dan template tidak sama.")
if train_df[TEXT_COL].str.len().eq(0).any():
    raise ValueError("Ada teks train kosong.")
if test_df[TEST_TEXT_COL].str.len().eq(0).any():
    raise ValueError("Ada teks test kosong.")

y = train_df[LABEL_COL].map(label2id).astype(int).values

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Template shape:", template_df.shape)
display(train_df[LABEL_COL].value_counts().reindex(LABELS))


## 05. Text Normalization and Features

In [ ]:
SLANG_MAP = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak", "tdk": "tidak", "tak": "tidak",
    "bgt": "banget", "yg": "yang", "utk": "untuk", "dgn": "dengan", "krn": "karena", "karna": "karena",
    "dr": "dari", "dlm": "dalam", "aja": "saja", "sm": "sama", "sdh": "sudah", "udah": "sudah", "blm": "belum",
    "dpt": "dapat", "dapet": "dapat", "jd": "jadi", "jgn": "jangan", "org": "orang", "pd": "pada", "tp": "tapi",
    "trs": "terus", "mnrt": "menurut", "hrs": "harus", "skrg": "sekarang", "klo": "kalau", "kl": "kalau", "emg": "memang",
    "kmrn": "kemarin", "bkn": "bukan", "krja": "kerja", "duitnya": "duit", "uangnya": "uang"
}

KEYWORDS = {
    "Anggaran": ["anggaran", "dana", "biaya", "apbn", "triliun", "miliar", "rupiah", "rp", "duit", "uang", "pajak", "alokasi", "efisiensi", "pembiayaan", "fiskal", "belanja", "subsidi", "utang", "boros", "korupsi"],
    "Distribusi": ["distribusi", "penyaluran", "dibagikan", "pembagian", "merata", "belum merata", "daerah", "wilayah", "pelosok", "desa", "kecamatan", "kabupaten", "provinsi", "logistik", "terjangkau", "sampai", "akses", "sekolah belum", "jadwal", "terlambat"],
    "Ekonomi": ["ekonomi", "umkm", "usaha mikro", "koperasi", "pemasok", "supplier", "vendor", "petani", "nelayan", "peternak", "pedagang", "pasar", "pangan lokal", "produk lokal", "rantai pasok", "supply chain", "lapangan kerja", "tenaga kerja", "usaha", "bisnis", "industri", "perputaran ekonomi", "ekonomi lokal", "pelaku usaha"],
    "Kualitas Pangan": ["makanan", "pangan", "gizi", "bergizi", "menu", "porsi", "susu", "nasi", "telur", "sayur", "lauk", "buah", "keracunan", "beracun", "belatung", "basi", "tidak layak", "higienis", "sehat", "sakit", "muntah", "diare", "bau", "kadaluwarsa", "nutrisi", "kalori", "kualitas makanan"],
    "Lainnya": ["wkwk", "hehe", "haha", "lol", "anjir", "anjay", "random", "event", "panitia", "mau mbg", "mbg apa", "my mbg", "user mbg", "is coming"],
    "Politik": ["prabowo", "gibran", "jokowi", "presiden", "wapres", "pemerintah", "menteri", "kampanye", "janji", "janji kampanye", "oposisi", "partai", "politik", "pilpres", "pemilu", "rezim", "koalisi", "kabinet", "rakyat", "program prabowo", "citra", "pencitraan", "dukungan", "kritik pemerintah", "buzzer", "gerindra"],
    "Sasaran Penerima": ["penerima", "penerima manfaat", "siswa", "anak", "anak sekolah", "balita", "ibu hamil", "ibu menyusui", "santri", "pelajar", "murid", "target", "sasaran", "stunting", "generasi", "gizi anak", "kelompok penerima", "cakupan", "jumlah penerima", "manfaat"],
    "Tata Kelola": ["tata kelola", "pengawasan", "evaluasi", "bgn", "sppg", "sop", "dapur", "regulasi", "koordinasi", "pelaksanaan", "implementasi", "audit", "standar", "prosedur", "mekanisme", "kementerian", "lembaga", "badan gizi", "pengelolaan", "pengawas", "monitoring", "mitigasi", "perbaikan sistem", "vendor", "mitra", "penyelenggara"]
}

STRONG_KEYWORDS = {
    "Anggaran": ["apbn", "triliun", "miliar", "anggaran", "pajak", "dana", "alokasi", "biaya", "boros"],
    "Distribusi": ["distribusi", "penyaluran", "merata", "pelosok", "logistik", "terjangkau", "belum merata"],
    "Ekonomi": ["umkm", "petani", "nelayan", "peternak", "pemasok", "supplier", "rantai pasok", "lapangan kerja", "ekonomi lokal", "produk lokal", "pangan lokal", "pelaku usaha"],
    "Kualitas Pangan": ["keracunan", "basi", "belatung", "muntah", "diare", "kadaluwarsa", "higienis", "beracun"],
    "Lainnya": ["wkwk", "haha", "hehe", "lol", "my mbg", "user mbg", "is coming"],
    "Politik": ["prabowo", "gibran", "jokowi", "kampanye", "janji kampanye", "oposisi", "partai", "pilpres", "pencitraan", "buzzer"],
    "Sasaran Penerima": ["penerima manfaat", "ibu hamil", "balita", "stunting", "anak sekolah", "siswa"],
    "Tata Kelola": ["sop", "pengawasan", "evaluasi", "bgn", "sppg", "regulasi", "koordinasi", "audit", "badan gizi", "prosedur", "dapur", "vendor"]
}


def normalize_text(text, lowercase=True):
    text = html.unescape(str(text))
    text = re.sub(r"http\S+|www\.\S+", " <URL> ", text)
    text = re.sub(r"@\w+", " <USER> ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"\bRp\s?([0-9\.,]+)", r" rupiah \1 ", text, flags=re.I)
    text = re.sub(r"([0-9]+)\s?(T|triliun)\b", r" \1 triliun ", text, flags=re.I)
    text = re.sub(r"([0-9]+)\s?(M|miliar)\b", r" \1 miliar ", text, flags=re.I)
    text = re.sub(r"(.)\1{3,}", r"\1\1", text)
    if lowercase:
        text = text.lower()
    tokens = []
    for tok in text.split():
        key = re.sub(r"[^\w<>]+", "", tok.lower())
        tokens.append(SLANG_MAP.get(key, tok))
    return re.sub(r"\s+", " ", " ".join(tokens)).strip()


def keyword_scores(text):
    t = str(text).lower()
    scores = []
    for lab in LABELS:
        sc = 0.0
        for kw in KEYWORDS.get(lab, []):
            if kw in t:
                sc += 1.0
        for kw in STRONG_KEYWORDS.get(lab, []):
            if kw in t:
                sc += 2.0
        scores.append(sc)
    return np.asarray(scores, dtype=float)


def make_hint_text(text):
    clean = normalize_text(text)
    scores = keyword_scores(clean)
    hints = []
    for lab, sc in zip(LABELS, scores):
        if lab == "Ekonomi" and sc >= 3:
            hints.append("[KW_EKONOMI]")
        elif lab in ["Politik", "Tata Kelola"] and sc >= 2:
            hints.append("[KW_" + lab.upper().replace(" ", "_") + "]")
        elif sc > 0:
            hints.append("[KW_" + lab.upper().replace(" ", "_") + "]")
    return " ".join(hints + [clean])


def text_variants_for_tta(text, use_hint=False):
    clean = normalize_text(text)
    hint = make_hint_text(text)
    raw_norm = normalize_text(text, lowercase=False)
    if use_hint:
        variants = [hint, clean]
    else:
        variants = [clean, hint, raw_norm]
    return list(dict.fromkeys([v for v in variants if isinstance(v, str) and v.strip()]))


def meta_text_features(texts):
    rows = []
    for text in texts:
        raw = str(text)
        clean = normalize_text(raw)
        ks = keyword_scores(clean)
        chars = len(raw)
        words = len(raw.split())
        mentions = len(re.findall(r"@\w+", raw))
        urls = len(re.findall(r"http\S+|www\.\S+", raw))
        hashtags = len(re.findall(r"#\w+", raw))
        qmarks = raw.count("?")
        emarks = raw.count("!")
        caps = sum(1 for c in raw if c.isupper())
        alpha = sum(1 for c in raw if c.isalpha())
        cap_ratio = caps / max(alpha, 1)
        digits = sum(1 for c in raw if c.isdigit())
        rows.append([chars, words, mentions, urls, hashtags, qmarks, emarks, cap_ratio, digits] + ks.tolist())
    cols = ["char_len", "word_count", "mention_count", "url_count", "hashtag_count", "question_count", "exclaim_count", "cap_ratio", "digit_count"] + [f"kw_{lab}" for lab in LABELS]
    return pd.DataFrame(rows, columns=cols)

train_df["text_clean"] = train_df[TEXT_COL].map(normalize_text)
train_df["text_hint"] = train_df[TEXT_COL].map(make_hint_text)
test_df["text_clean"] = test_df[TEST_TEXT_COL].map(normalize_text)
test_df["text_hint"] = test_df[TEST_TEXT_COL].map(make_hint_text)
train_meta = meta_text_features(train_df[TEXT_COL].tolist())
test_meta = meta_text_features(test_df[TEST_TEXT_COL].tolist())

display(train_df[[TEXT_COL, "text_clean", "text_hint", LABEL_COL]].head(3))


## 06. Evaluation Utilities

In [ ]:
def normalize_prob(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.nan_to_num(p, nan=1.0 / NUM_LABELS, posinf=1.0, neginf=0.0)
    p = np.clip(p, 1e-12, None)
    return p / p.sum(axis=1, keepdims=True)


def pred_from_prob(prob):
    return np.argmax(normalize_prob(prob), axis=1)


def entropy_prob(prob):
    p = normalize_prob(prob)
    return -np.sum(p * np.log(p + 1e-12), axis=1)


def margin_prob(prob):
    p = normalize_prob(prob)
    s = np.sort(p, axis=1)
    return s[:, -1] - s[:, -2]


def floor_aware_score(prob, y_true, floors=RECALL_FLOORS, alpha=0.35):
    pred = pred_from_prob(prob)
    ba = balanced_accuracy_score(y_true, pred)
    rec = recall_score(y_true, pred, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    penalty = 0.0
    for lab, floor in floors.items():
        penalty += max(0.0, floor - rec[label2id[lab]])
    return ba - alpha * penalty


def evaluate_prob(y_true, prob, title):
    pred = pred_from_prob(prob)
    ba = balanced_accuracy_score(y_true, pred)
    rec = recall_score(y_true, pred, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    print(f"\n{title}")
    print(f"Balanced Accuracy: {ba:.6f}")
    display(pd.DataFrame({"label": LABELS, "recall": rec, "support": np.bincount(y_true, minlength=NUM_LABELS)}))
    return ba, rec


def save_report(y_true, prob, path):
    pred = pred_from_prob(prob)
    rep = classification_report(y_true, pred, target_names=LABELS, output_dict=True, zero_division=0)
    pd.DataFrame(rep).T.to_excel(path)


def show_confusion(y_true, prob):
    cm = confusion_matrix(y_true, pred_from_prob(prob), labels=list(range(NUM_LABELS)))
    cm_df = pd.DataFrame(cm, index=["true_" + x for x in LABELS], columns=["pred_" + x for x in LABELS])
    display(cm_df)
    return cm_df


def ensure_prob_shape(p, classes=None, n_classes=NUM_LABELS):
    p = np.asarray(p, dtype=np.float64)
    if p.ndim == 1:
        p = np.column_stack([1.0 - p, p])
    if p.shape[1] == n_classes:
        return normalize_prob(p)
    out = np.zeros((p.shape[0], n_classes), dtype=np.float64)
    if classes is None:
        classes = np.arange(p.shape[1])
    for j, cls in enumerate(classes):
        cls = int(cls)
        if 0 <= cls < n_classes:
            out[:, cls] = p[:, j]
    return normalize_prob(out)


## 07. TF-IDF Feature Pipeline

In [ ]:
def build_sparse_features(train_texts, valid_texts=None, test_texts=None, train_meta_df=None, valid_meta_df=None, test_meta_df=None):
    word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.98, max_features=180000, sublinear_tf=True, strip_accents="unicode", dtype=np.float32)
    char_wb_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=2, max_features=260000, sublinear_tf=True, dtype=np.float32)
    char_vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2, max_features=180000, sublinear_tf=True, dtype=np.float32)
    Xw_train = word_vec.fit_transform(train_texts)
    Xcw_train = char_wb_vec.fit_transform(train_texts)
    Xc_train = char_vec.fit_transform(train_texts)
    scaler = StandardScaler(with_mean=False)
    Xm_train = sparse.csr_matrix(train_meta_df.values.astype(np.float32)) if train_meta_df is not None else None
    parts = [Xw_train, Xcw_train, Xc_train]
    if Xm_train is not None:
        Xm_train = scaler.fit_transform(Xm_train)
        parts.append(Xm_train)
    X_train = sparse.hstack(parts, format="csr")
    outputs = {"X_train": X_train, "word_vec": word_vec, "char_wb_vec": char_wb_vec, "char_vec": char_vec, "scaler": scaler}

    def transform(texts, meta_df):
        Xw = word_vec.transform(texts)
        Xcw = char_wb_vec.transform(texts)
        Xc = char_vec.transform(texts)
        parts = [Xw, Xcw, Xc]
        if meta_df is not None:
            Xm = sparse.csr_matrix(meta_df.values.astype(np.float32))
            Xm = scaler.transform(Xm)
            parts.append(Xm)
        return sparse.hstack(parts, format="csr")

    if valid_texts is not None:
        outputs["X_valid"] = transform(valid_texts, valid_meta_df)
    if test_texts is not None:
        outputs["X_test"] = transform(test_texts, test_meta_df)
    return outputs


def safe_predict_proba(clf, X):
    if hasattr(clf, "predict_proba"):
        return clf.predict_proba(X)
    if hasattr(clf, "decision_function"):
        d = clf.decision_function(X)
        if d.ndim == 1:
            d = np.column_stack([-d, d])
        return softmax(d, axis=1)
    pred = clf.predict(X)
    p = np.zeros((X.shape[0], NUM_LABELS), dtype=np.float64)
    p[np.arange(X.shape[0]), pred] = 1.0
    return p


def fit_model(clf, X, y_fit, sample_weight=None, name="model"):
    if sample_weight is None:
        return clf.fit(X, y_fit)
    try:
        return clf.fit(X, y_fit, sample_weight=sample_weight)
    except Exception as e:
        msg = str(e).lower()
        if "sample_weight" in msg or "metadata" in msg or isinstance(e, TypeError):
            print(f"{name}: sample_weight tidak diterima, retry tanpa sample_weight")
            return clf.fit(X, y_fit)
        raise


def make_calibrated_linear_svc():
    base = LinearSVC(C=0.8, class_weight="balanced", max_iter=6000, random_state=SEED)
    try:
        return CalibratedClassifierCV(estimator=base, method="sigmoid", cv=3)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=base, method="sigmoid", cv=3)


def get_tfidf_classifiers():
    return [
        ("lr_saga_c18", LogisticRegression(C=1.8, class_weight="balanced", solver="saga", max_iter=5000, n_jobs=-1, random_state=SEED)),
        ("lr_saga_c32", LogisticRegression(C=3.2, class_weight="balanced", solver="saga", max_iter=5000, n_jobs=-1, random_state=SEED)),
        ("lr_ovr_c26", OneVsRestClassifier(LogisticRegression(C=2.6, class_weight="balanced", solver="liblinear", max_iter=4000, random_state=SEED))),
        ("sgd_log", SGDClassifier(loss="log_loss", alpha=2e-5, penalty="elasticnet", l1_ratio=0.08, class_weight="balanced", max_iter=3500, random_state=SEED)),
        ("ridge_a10", RidgeClassifier(alpha=1.0, class_weight="balanced", random_state=SEED)),
        ("ridge_a30", RidgeClassifier(alpha=3.0, class_weight="balanced", random_state=SEED)),
        ("cnb_a015", ComplementNB(alpha=0.15)),
        ("cnb_a075", ComplementNB(alpha=0.75)),
        ("linsvc_cal", make_calibrated_linear_svc()),
    ]


def class_balanced_sample_weights(y_train, extra_weights=None):
    classes = np.arange(NUM_LABELS)
    present = np.unique(y_train)
    cw_map = np.ones(NUM_LABELS, dtype=np.float64)
    cw = compute_class_weight(class_weight="balanced", classes=present, y=y_train)
    for cls, val in zip(present, cw):
        cw_map[int(cls)] = float(val)
    weights = cw_map[y_train].astype(np.float64)
    emphasis = {"Ekonomi": 1.15, "Distribusi": 1.08, "Sasaran Penerima": 1.08, "Tata Kelola": 1.12, "Politik": 1.08, "Lainnya": 1.05}
    for lab, mult in emphasis.items():
        weights[y_train == label2id[lab]] *= mult
    if extra_weights is not None:
        weights *= np.asarray(extra_weights, dtype=np.float64)
    weights = weights / max(weights.mean(), 1e-12)
    return weights


def compute_noise_weights(y_true, prob_list):
    avg = normalize_prob(np.mean(prob_list, axis=0))
    true_conf = avg[np.arange(len(y_true)), y_true]
    pred = pred_from_prob(avg)
    err = (pred != y_true).astype(float)
    weight = np.ones(len(y_true), dtype=np.float64)
    weight[true_conf < 0.20] *= 0.70
    weight[(true_conf < 0.12) & (err == 1)] *= 0.75
    weight = np.clip(weight, 0.45, 1.10)
    return weight, true_conf


## 08. Run TF-IDF OOF

In [ ]:
def run_tfidf_oof(n_splits=N_SPLITS_TFIDF, text_col="text_clean", extra_sample_weight=None, title="tfidf"):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    model_names = [name for name, _ in get_tfidf_classifiers()]
    oof_by_model = {name: np.zeros((len(train_df), NUM_LABELS), dtype=np.float64) for name in model_names}
    test_by_model = {name: np.zeros((len(test_df), NUM_LABELS), dtype=np.float64) for name in model_names}
    test_texts = test_df[text_col].tolist()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y), 1):
        print(f"\n{title} fold {fold}/{n_splits}")
        feats = build_sparse_features(
            train_df.iloc[tr_idx][text_col].tolist(),
            train_df.iloc[va_idx][text_col].tolist(),
            test_texts,
            train_meta.iloc[tr_idx].reset_index(drop=True),
            train_meta.iloc[va_idx].reset_index(drop=True),
            test_meta
        )
        ytr = y[tr_idx]
        sw = class_balanced_sample_weights(ytr, extra_sample_weight[tr_idx] if extra_sample_weight is not None else None)
        for name, clf in get_tfidf_classifiers():
            print("training", name)
            try:
                fit_model(clf, feats["X_train"], ytr, sw, name)
                pva = safe_predict_proba(clf, feats["X_valid"])
                pte = safe_predict_proba(clf, feats["X_test"])
                classes = getattr(clf, "classes_", None)
                if classes is None and hasattr(clf, "estimator"):
                    classes = getattr(clf.estimator, "classes_", None)
                pva = ensure_prob_shape(pva, classes)
                pte = ensure_prob_shape(pte, classes)
                oof_by_model[name][va_idx] = pva
                test_by_model[name] += pte / n_splits
                print(f"fold BA {balanced_accuracy_score(y[va_idx], pred_from_prob(pva)):.5f}")
            except Exception as e:
                print(f"model {name} gagal pada fold {fold}:", repr(e))
        gc.collect()
    return oof_by_model, {k: normalize_prob(v) for k, v in test_by_model.items()}

tfidf_oof_by_model, tfidf_test_by_model = {}, {}
if RUN_TFIDF:
    tfidf_oof_by_model, tfidf_test_by_model = run_tfidf_oof(title="tfidf_initial")
    tfidf_avg_oof = normalize_prob(np.mean(list(tfidf_oof_by_model.values()), axis=0))
    evaluate_prob(y, tfidf_avg_oof, "TF-IDF initial average")
    if RUN_TFIDF_NOISE_RETRAIN:
        noise_weight, true_conf = compute_noise_weights(y, list(tfidf_oof_by_model.values()))
        train_df["noise_weight"] = noise_weight
        train_df["oof_true_conf_tfidf"] = true_conf
        train_df.assign(pred=[id2label[i] for i in pred_from_prob(tfidf_avg_oof)]).to_excel(OUTPUT_DIR / "oof_tfidf_diagnostics.xlsx", index=False)
        retr_oof, retr_test = run_tfidf_oof(extra_sample_weight=noise_weight, title="tfidf_noise_retrain")
        for k, v in retr_oof.items():
            tfidf_oof_by_model[f"noise_{k}"] = v
        for k, v in retr_test.items():
            tfidf_test_by_model[f"noise_{k}"] = v
    tfidf_avg_oof = normalize_prob(np.mean(list(tfidf_oof_by_model.values()), axis=0))
    tfidf_avg_test = normalize_prob(np.mean(list(tfidf_test_by_model.values()), axis=0))
    evaluate_prob(y, tfidf_avg_oof, "TF-IDF all average")
else:
    tfidf_avg_oof = np.ones((len(train_df), NUM_LABELS), dtype=np.float64) / NUM_LABELS
    tfidf_avg_test = np.ones((len(test_df), NUM_LABELS), dtype=np.float64) / NUM_LABELS


## 09. Binary Gates and Pairwise Specialists

In [ ]:
def run_binary_gates(labels_to_gate=None, n_splits=N_SPLITS_BINARY):
    if labels_to_gate is None:
        labels_to_gate = ["Ekonomi", "Distribusi", "Sasaran Penerima", "Tata Kelola", "Politik", "Lainnya"]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    gate_oof = {lab: np.zeros(len(train_df), dtype=np.float64) for lab in labels_to_gate}
    gate_test = {lab: np.zeros(len(test_df), dtype=np.float64) for lab in labels_to_gate}
    for lab in labels_to_gate:
        print(f"\nBinary gate {lab}")
        yy = (y == label2id[lab]).astype(int)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, yy), 1):
            feats = build_sparse_features(
                train_df.iloc[tr_idx]["text_clean"].tolist(),
                train_df.iloc[va_idx]["text_clean"].tolist(),
                test_df["text_clean"].tolist(),
                train_meta.iloc[tr_idx].reset_index(drop=True),
                train_meta.iloc[va_idx].reset_index(drop=True),
                test_meta
            )
            clf = LogisticRegression(C=2.0, class_weight="balanced", solver="liblinear", max_iter=3000, random_state=SEED)
            clf.fit(feats["X_train"], yy[tr_idx])
            pva = safe_predict_proba(clf, feats["X_valid"])
            pte = safe_predict_proba(clf, feats["X_test"])
            cls = getattr(clf, "classes_", [0, 1])
            pos_col = list(cls).index(1) if 1 in cls else min(1, pva.shape[1] - 1)
            gate_oof[lab][va_idx] = pva[:, pos_col]
            gate_test[lab] += pte[:, pos_col] / n_splits
    return gate_oof, gate_test


def run_pairwise_specialists(n_splits=N_SPLITS_PAIRWISE):
    pairs = [("Politik", "Anggaran"), ("Politik", "Kualitas Pangan"), ("Politik", "Tata Kelola"), ("Tata Kelola", "Kualitas Pangan"), ("Distribusi", "Sasaran Penerima"), ("Ekonomi", "Tata Kelola"), ("Ekonomi", "Anggaran"), ("Lainnya", "Politik"), ("Lainnya", "Kualitas Pangan")]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    pair_oof, pair_test = {}, {}
    for a, b in pairs:
        print(f"\nPairwise {a} vs {b}")
        ia, ib = label2id[a], label2id[b]
        yy = np.full(len(y), -1, dtype=int)
        yy[y == ia] = 0
        yy[y == ib] = 1
        pair_oof[(a, b)] = np.zeros((len(train_df), 2), dtype=np.float64)
        pair_test[(a, b)] = np.zeros((len(test_df), 2), dtype=np.float64)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y), 1):
            tr_loc = tr_idx[yy[tr_idx] >= 0]
            if len(np.unique(yy[tr_loc])) < 2 or len(tr_loc) < 20:
                continue
            feats = build_sparse_features(
                train_df.iloc[tr_loc]["text_clean"].tolist(),
                train_df.iloc[va_idx]["text_clean"].tolist(),
                test_df["text_clean"].tolist(),
                train_meta.iloc[tr_loc].reset_index(drop=True),
                train_meta.iloc[va_idx].reset_index(drop=True),
                test_meta
            )
            clf = LogisticRegression(C=2.0, class_weight="balanced", solver="liblinear", max_iter=3000, random_state=SEED)
            clf.fit(feats["X_train"], yy[tr_loc])
            pva = safe_predict_proba(clf, feats["X_valid"])
            pte = safe_predict_proba(clf, feats["X_test"])
            fva = ensure_prob_shape(pva, getattr(clf, "classes_", [0, 1]), n_classes=2)
            fte = ensure_prob_shape(pte, getattr(clf, "classes_", [0, 1]), n_classes=2)
            pair_oof[(a, b)][va_idx] = fva
            pair_test[(a, b)] += fte / n_splits
    return pair_oof, pair_test

gate_oof, gate_test = {}, {}
pair_oof, pair_test = {}, {}
if RUN_BINARY_GATES:
    gate_oof, gate_test = run_binary_gates()
if RUN_PAIRWISE:
    pair_oof, pair_test = run_pairwise_specialists()


## 10. Transformer Dataset and Training Functions

In [ ]:
if TRANSFORMERS_AVAILABLE:
    class MBGDataset(Dataset):
        def __init__(self, texts, labels=None, tokenizer=None, max_len=192, sample_weights=None):
            self.texts = list(texts)
            self.labels = None if labels is None else np.asarray(labels, dtype=np.int64)
            self.tokenizer = tokenizer
            self.max_len = max_len
            self.sample_weights = None if sample_weights is None else np.asarray(sample_weights, dtype=np.float32)

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding="max_length", max_length=self.max_len, return_tensors=None)
            item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            if self.sample_weights is not None:
                item["sample_weight"] = torch.tensor(self.sample_weights[idx], dtype=torch.float32)
            return item

    class TAPTDataset(Dataset):
        def __init__(self, texts, tokenizer, max_len=192):
            self.texts = [str(t) for t in texts if str(t).strip()]
            self.tokenizer = tokenizer
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            return self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_len, return_special_tokens_mask=True)


def amp_context():
    if TRANSFORMERS_AVAILABLE and USE_AMP and DEVICE == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return nullcontext()


def make_loader(dataset, batch_size, shuffle=False, sampler=None, collate_fn=None):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle if sampler is None else False, sampler=sampler, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY and DEVICE == "cuda", persistent_workers=(NUM_WORKERS > 0 and PERSISTENT_WORKERS))


def add_special_tokens(tokenizer, model=None):
    if not ADD_SPECIAL_KEYWORD_TOKENS:
        return 0
    old_len = len(tokenizer)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    added = len(tokenizer) - old_len
    if model is not None and added > 0:
        model.resize_token_embeddings(len(tokenizer))
    return added


def class_weights_tensor(y_train):
    present = np.unique(y_train)
    cw_map = np.ones(NUM_LABELS, dtype=np.float32)
    cw = compute_class_weight(class_weight="balanced", classes=present, y=y_train)
    for cls, val in zip(present, cw):
        cw_map[int(cls)] = float(val)
    for lab, mult in {"Ekonomi": 1.20, "Distribusi": 1.08, "Sasaran Penerima": 1.08, "Tata Kelola": 1.12, "Politik": 1.08, "Lainnya": 1.05}.items():
        cw_map[label2id[lab]] *= mult
    cw_map = cw_map / cw_map.mean()
    return torch.tensor(cw_map, dtype=torch.float32, device=DEVICE), cw_map


def make_weighted_sampler(y_train):
    _, cw_np = class_weights_tensor(y_train)
    weights = cw_np[y_train]
    return WeightedRandomSampler(weights=torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)


def transformer_sample_weights(y_train, original_idx, cfg):
    weights = np.ones(len(y_train), dtype=np.float32)
    if cfg.get("use_noise_weight", True) and "noise_weight" in train_df.columns:
        weights *= train_df.iloc[original_idx]["noise_weight"].values.astype(np.float32)
    if cfg.get("minority_sample_weight", False):
        for lab, mult in {"Ekonomi": 1.10, "Tata Kelola": 1.06, "Distribusi": 1.05, "Sasaran Penerima": 1.05}.items():
            weights[y_train == label2id[lab]] *= mult
    weights = np.clip(weights, 0.45, 1.15)
    weights = weights / max(float(weights.mean()), 1e-6)
    return weights.astype(np.float32)


def build_optimizer(model, lr, head_lr):
    head_keys = ["classifier", "score", "pre_classifier"]
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    groups = []
    base_decay, base_nodecay, head_decay, head_nodecay = [], [], [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        is_head = any(k in name for k in head_keys)
        is_no_decay = any(k in name for k in no_decay)
        if is_head and is_no_decay:
            head_nodecay.append(param)
        elif is_head:
            head_decay.append(param)
        elif is_no_decay:
            base_nodecay.append(param)
        else:
            base_decay.append(param)
    if base_decay:
        groups.append({"params": base_decay, "lr": lr, "weight_decay": WEIGHT_DECAY})
    if base_nodecay:
        groups.append({"params": base_nodecay, "lr": lr, "weight_decay": 0.0})
    if head_decay:
        groups.append({"params": head_decay, "lr": head_lr, "weight_decay": WEIGHT_DECAY})
    if head_nodecay:
        groups.append({"params": head_nodecay, "lr": head_lr, "weight_decay": 0.0})
    return torch.optim.AdamW(groups)


def compute_loss(logits, labels, class_w=None, class_count_log=None, loss_mode="class_weight", sample_weight=None, label_smoothing=None):
    if label_smoothing is None:
        label_smoothing = LABEL_SMOOTHING if USE_LABEL_SMOOTHING else 0.0
    if loss_mode == "balanced_softmax" and class_count_log is not None:
        logits = logits + class_count_log.view(1, -1)
    weight = class_w if loss_mode in {"class_weight", "balanced_softmax"} else None
    losses = F.cross_entropy(logits, labels, weight=weight, label_smoothing=label_smoothing, reduction="none")
    if sample_weight is not None:
        losses = losses * sample_weight.to(losses.device).float()
    return losses.mean()


def predict_transformer(model, tokenizer, texts, max_len, batch_size, tta=False, use_hint=False):
    model.eval()
    all_probs = []
    variants = []
    if tta:
        for text in texts:
            variants.append(text_variants_for_tta(text, use_hint=use_hint))
        flat = [v for row in variants for v in row]
        ds = MBGDataset(flat, None, tokenizer, max_len)
    else:
        ds = MBGDataset(texts, None, tokenizer, max_len)
    loader = make_loader(ds, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with amp_context():
                logits = model(**batch).logits
            all_probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    probs = np.vstack(all_probs)
    if tta:
        out = np.zeros((len(texts), NUM_LABELS), dtype=np.float64)
        cursor = 0
        for i, row in enumerate(variants):
            n = len(row)
            out[i] = probs[cursor:cursor + n].mean(axis=0)
            cursor += n
        probs = out
    return normalize_prob(probs)


def load_state_dict_safe(model, path):
    try:
        state = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)


def safe_alias_name(name):
    return re.sub(r"[^a-zA-Z0-9_]+", "_", str(name)).strip("_")


## 11B. TAPT Train-Only Domain Adaptation

In [ ]:
TAPT_MODEL_PATHS = {}

def run_tapt_train_only(model_name):
    if not (RUN_TAPT and RUN_TRANSFORMERS and TRANSFORMERS_AVAILABLE and DEVICE == "cuda"):
        return model_name
    out_dir = CHECKPOINT_DIR / f"{CACHE_VERSION}_tapt_{safe_alias_name(model_name)}"
    if (out_dir / "config.json").exists() and (out_dir / "tokenizer_config.json").exists():
        print("TAPT cache ditemukan:", out_dir)
        return str(out_dir)
    print(f"\n=== TAPT train-only | {model_name} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    add_special_tokens(tokenizer, model)
    if getattr(tokenizer, "mask_token", None) is None:
        print("Tokenizer tidak memiliki mask token, TAPT dilewati:", model_name)
        return model_name
    model.to(DEVICE)
    if USE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False
    texts = train_df["text_clean"].dropna().astype(str).tolist()
    if TAPT_USE_TEST_TEXT:
        texts = texts + test_df["text_clean"].dropna().astype(str).tolist()
    dataset = TAPTDataset(texts, tokenizer, TAPT_MAX_LEN)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=TAPT_MLM_PROB)
    loader = make_loader(dataset, batch_size=TAPT_BATCH_SIZE, shuffle=True, collate_fn=collator)
    optimizer = torch.optim.AdamW(model.parameters(), lr=TAPT_LR, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, TAPT_EPOCHS * len(loader))
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda" and not USE_BF16))
    for epoch in range(1, TAPT_EPOCHS + 1):
        model.train()
        losses = []
        pbar = tqdm(loader, desc=f"TAPT {safe_alias_name(model_name)} e{epoch}")
        for batch in pbar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))
            pbar.set_postfix(loss=np.mean(losses[-20:]))
        print(f"TAPT epoch {epoch} loss={np.mean(losses):.5f}")
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    del model, dataset, loader
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("TAPT saved:", out_dir)
    return str(out_dir)

if RUN_TAPT and RUN_TRANSFORMERS and TRANSFORMERS_AVAILABLE and DEVICE == "cuda":
    tapt_targets = sorted({cfg["model_name"] for cfg in TRANSFORMER_MODEL_CONFIGS if cfg.get("use_tapt", False)})
    for model_name in tapt_targets:
        try:
            TAPT_MODEL_PATHS[model_name] = run_tapt_train_only(model_name)
        except Exception as e:
            print("TAPT gagal, fallback ke base model:", model_name, repr(e))
            TAPT_MODEL_PATHS[model_name] = model_name
else:
    print("TAPT dilewati.")


## 11. Run Transformer OOF

In [ ]:
def run_one_transformer_config(cfg):
    alias = cfg["alias"]
    seed_everything(cfg.get("seed", SEED))
    use_hint = bool(cfg.get("use_hint", False))
    text_col = "text_hint" if use_hint else "text_clean"
    base_model_name = cfg["model_name"]
    model_name = TAPT_MODEL_PATHS.get(base_model_name, base_model_name) if cfg.get("use_tapt", False) else base_model_name
    max_len = cfg.get("max_len", 192)
    batch_size = cfg.get("batch_size", 16)
    eval_batch_size = cfg.get("eval_batch_size", 48)
    epochs = cfg.get("epochs", 4)
    lr = cfg.get("lr", 1.5e-5)
    head_lr = cfg.get("head_lr", lr * 3)
    loss_mode = cfg.get("loss_mode", "class_weight")
    use_sampler = bool(cfg.get("use_sampler", False))
    use_noise_weight = bool(cfg.get("use_noise_weight", True))
    print(f"\n=== {alias} | {model_name} | text={text_col} | loss={loss_mode} | sampler={use_sampler} | noise_w={use_noise_weight} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    skf = StratifiedKFold(n_splits=N_SPLITS_TRANSFORMER, shuffle=True, random_state=cfg.get("seed", SEED))
    oof = np.zeros((len(train_df), NUM_LABELS), dtype=np.float64)
    test_prob = np.zeros((len(test_df), NUM_LABELS), dtype=np.float64)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y), 1):
        print(f"\n{alias} fold {fold}/{N_SPLITS_TRANSFORMER}")
        fold_path = CHECKPOINT_DIR / f"{CACHE_VERSION}_{alias}_fold{fold}.pt"
        tr_texts = train_df.iloc[tr_idx][text_col].tolist()
        va_texts = train_df.iloc[va_idx][text_col].tolist()
        te_texts = test_df[text_col].tolist()
        ytr, yva = y[tr_idx], y[va_idx]
        sw = transformer_sample_weights(ytr, tr_idx, cfg) if use_noise_weight else None
        train_ds = MBGDataset(tr_texts, ytr, tokenizer, max_len, sample_weights=sw)
        valid_ds = MBGDataset(va_texts, yva, tokenizer, max_len)
        sampler = make_weighted_sampler(ytr) if use_sampler else None
        train_loader = make_loader(train_ds, batch_size=batch_size, shuffle=True, sampler=sampler)
        valid_loader = make_loader(valid_ds, batch_size=eval_batch_size, shuffle=False)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
        add_special_tokens(tokenizer, model)
        model.to(DEVICE)
        if USE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
            if hasattr(model.config, "use_cache"):
                model.config.use_cache = False
        optimizer = build_optimizer(model, lr, head_lr)
        total_steps = max(1, epochs * len(train_loader))
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        class_w, cw_np = class_weights_tensor(ytr)
        count = np.bincount(ytr, minlength=NUM_LABELS).astype(np.float32)
        class_count_log = torch.log(torch.tensor(np.maximum(count, 1.0), device=DEVICE, dtype=torch.float32))
        scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda" and not USE_BF16))
        best_score = -1e9
        best_ba = -1e9
        for epoch in range(1, epochs + 1):
            model.train()
            losses = []
            pbar = tqdm(train_loader, desc=f"{alias} f{fold} e{epoch}")
            for batch in pbar:
                labels = batch.pop("labels").to(DEVICE)
                sample_weight = batch.pop("sample_weight", None)
                if sample_weight is not None:
                    sample_weight = sample_weight.to(DEVICE)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                optimizer.zero_grad(set_to_none=True)
                with amp_context():
                    logits = model(**batch).logits
                    loss = compute_loss(logits.float(), labels, class_w=class_w, class_count_log=class_count_log, loss_mode=loss_mode, sample_weight=sample_weight)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                losses.append(float(loss.detach().cpu()))
                pbar.set_postfix(loss=np.mean(losses[-20:]))
            val_probs = []
            val_labels = []
            model.eval()
            with torch.no_grad():
                for batch in valid_loader:
                    labels = batch.pop("labels").numpy()
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    with amp_context():
                        logits = model(**batch).logits
                    val_probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
                    val_labels.append(labels)
            val_prob = normalize_prob(np.vstack(val_probs))
            val_true = np.concatenate(val_labels)
            ba = balanced_accuracy_score(val_true, pred_from_prob(val_prob))
            score = floor_aware_score(val_prob, val_true)
            print(f"epoch {epoch} loss={np.mean(losses):.5f} BA={ba:.6f} floor_score={score:.6f}")
            if score > best_score:
                best_score = score
                best_ba = ba
                torch.save(model.state_dict(), fold_path)
        load_state_dict_safe(model, fold_path)
        pva = predict_transformer(model, tokenizer, va_texts, max_len, eval_batch_size, tta=USE_TTA, use_hint=use_hint)
        pte = predict_transformer(model, tokenizer, te_texts, max_len, eval_batch_size, tta=USE_TTA, use_hint=use_hint)
        oof[va_idx] = pva
        test_prob += pte / N_SPLITS_TRANSFORMER
        print(f"best fold BA={best_ba:.6f} final fold BA={balanced_accuracy_score(yva, pred_from_prob(pva)):.6f}")
        del model, train_ds, valid_ds, train_loader, valid_loader
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    evaluate_prob(y, oof, alias)
    return normalize_prob(oof), normalize_prob(test_prob)

transformer_oof_by_model, transformer_test_by_model = {}, {}
if RUN_TRANSFORMERS and TRANSFORMERS_AVAILABLE and DEVICE == "cuda":
    for cfg in TRANSFORMER_MODEL_CONFIGS:
        try:
            oof_p, test_p = run_one_transformer_config(cfg)
            transformer_oof_by_model[cfg["alias"]] = oof_p
            transformer_test_by_model[cfg["alias"]] = test_p
        except RuntimeError as e:
            msg = str(e).lower()
            print(f"Transformer {cfg['alias']} gagal:", repr(e))
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            if "out of memory" in msg or "cuda" in msg:
                retry_cfg = cfg.copy()
                retry_cfg["alias"] = cfg["alias"] + "_retry_halfbatch"
                retry_cfg["batch_size"] = max(4, int(cfg.get("batch_size", 16)) // 2)
                retry_cfg["eval_batch_size"] = max(8, int(cfg.get("eval_batch_size", 48)) // 2)
                print("Retry dengan batch lebih kecil:", retry_cfg)
                try:
                    oof_p, test_p = run_one_transformer_config(retry_cfg)
                    transformer_oof_by_model[retry_cfg["alias"]] = oof_p
                    transformer_test_by_model[retry_cfg["alias"]] = test_p
                except Exception as e2:
                    print(f"Retry transformer {retry_cfg['alias']} tetap gagal:", repr(e2))
                    gc.collect()
                    if DEVICE == "cuda":
                        torch.cuda.empty_cache()
        except Exception as e:
            print(f"Transformer {cfg['alias']} gagal:", repr(e))
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
elif RUN_TRANSFORMERS:
    print("Transformer dilewati karena CUDA/transformers tidak tersedia.")


## 12. Model Pool and Meta Stacking

In [ ]:
def build_model_pool():
    pool = []
    for name, oof_p in tfidf_oof_by_model.items():
        if name in tfidf_test_by_model:
            pool.append({"name": "tfidf_" + name, "oof": normalize_prob(oof_p), "test": normalize_prob(tfidf_test_by_model[name])})
    for name, oof_p in transformer_oof_by_model.items():
        if name in transformer_test_by_model:
            pool.append({"name": "tr_" + name, "oof": normalize_prob(oof_p), "test": normalize_prob(transformer_test_by_model[name])})
    if not pool:
        raise RuntimeError("Tidak ada model valid di pool.")
    return pool


def summarize_pool(pool, title):
    rows = []
    for m in pool:
        pred = pred_from_prob(m["oof"])
        ba = balanced_accuracy_score(y, pred)
        rec = recall_score(y, pred, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
        rows.append({"name": m["name"], "BA": ba, "floor_score": floor_aware_score(m["oof"], y), **{f"rec_{lab}": rec[label2id[lab]] for lab in LABELS}})
    df = pd.DataFrame(rows).sort_values(["floor_score", "BA"], ascending=False).reset_index(drop=True)
    print(title)
    display(df)
    return df


def reject_weak_models(pool):
    df = summarize_pool(pool, "Raw model pool")
    best = float(df["floor_score"].max())
    keep = set(df.loc[(df["BA"] >= 0.50) & (df["floor_score"] >= best - 0.08), "name"])
    for nm in df.head(min(3, len(df)))["name"]:
        keep.add(nm)
    new_pool = [m for m in pool if m["name"] in keep]
    print("Kept models:", [m["name"] for m in new_pool])
    df.to_excel(OUTPUT_DIR / "model_pool_before_rejection.xlsx", index=False)
    summarize_pool(new_pool, "Model pool after rejection").to_excel(OUTPUT_DIR / "model_pool_after_rejection.xlsx", index=False)
    return new_pool


def meta_feature_matrix(pool, base_oof, base_test):
    Xo = [m["oof"] for m in pool]
    Xt = [m["test"] for m in pool]
    def conf_feats(p):
        p = normalize_prob(p)
        return np.column_stack([p.max(axis=1), margin_prob(p), entropy_prob(p), pred_from_prob(p)])
    Xo.append(conf_feats(base_oof))
    Xt.append(conf_feats(base_test))
    if gate_oof:
        labs = sorted(gate_oof.keys())
        Xo.append(np.column_stack([gate_oof[l] for l in labs]))
        Xt.append(np.column_stack([gate_test[l] for l in labs]))
    Xo.append(train_meta.values.astype(np.float32))
    Xt.append(test_meta.values.astype(np.float32))
    return np.hstack(Xo), np.hstack(Xt)


def run_lr_meta(pool, base_oof, base_test):
    X, Xt = meta_feature_matrix(pool, base_oof, base_test)
    skf = StratifiedKFold(n_splits=N_SPLITS_META, shuffle=True, random_state=SEED)
    oof = np.zeros((len(train_df), NUM_LABELS), dtype=np.float64)
    testp = np.zeros((len(test_df), NUM_LABELS), dtype=np.float64)
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        clf = LogisticRegression(C=1.2, class_weight="balanced", solver="lbfgs", max_iter=3000, random_state=SEED + fold)
        clf.fit(X[tr], y[tr])
        oof[va] = ensure_prob_shape(clf.predict_proba(X[va]), clf.classes_)
        testp += ensure_prob_shape(clf.predict_proba(Xt), clf.classes_) / N_SPLITS_META
    return normalize_prob(oof), normalize_prob(testp)


def run_lgbm_meta(pool, base_oof, base_test):
    if not LGBM_AVAILABLE:
        return None, None
    X, Xt = meta_feature_matrix(pool, base_oof, base_test)
    skf = StratifiedKFold(n_splits=N_SPLITS_META, shuffle=True, random_state=SEED)
    oof = np.zeros((len(train_df), NUM_LABELS), dtype=np.float64)
    testp = np.zeros((len(test_df), NUM_LABELS), dtype=np.float64)
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        clf = LGBMClassifier(objective="multiclass", num_class=NUM_LABELS, n_estimators=500, learning_rate=0.025, num_leaves=31, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.05, reg_lambda=0.3, class_weight="balanced", random_state=SEED + fold, verbosity=-1)
        clf.fit(X[tr], y[tr])
        oof[va] = ensure_prob_shape(clf.predict_proba(X[va]), clf.classes_)
        testp += ensure_prob_shape(clf.predict_proba(Xt), clf.classes_) / N_SPLITS_META
    return normalize_prob(oof), normalize_prob(testp)

model_pool = reject_weak_models(build_model_pool())
base_oof = normalize_prob(np.mean([m["oof"] for m in model_pool], axis=0))
base_test = normalize_prob(np.mean([m["test"] for m in model_pool], axis=0))
evaluate_prob(y, base_oof, "Base average ensemble")

if RUN_META_STACKING and len(model_pool) >= 2:
    lr_meta_oof, lr_meta_test = run_lr_meta(model_pool, base_oof, base_test)
    if floor_aware_score(lr_meta_oof, y) >= floor_aware_score(base_oof, y) - 0.005:
        model_pool.append({"name": "lr_meta", "oof": lr_meta_oof, "test": lr_meta_test})
        evaluate_prob(y, lr_meta_oof, "LR meta")
    if RUN_LGBM_META:
        lgb_oof, lgb_test = run_lgbm_meta(model_pool, base_oof, base_test)
        if lgb_oof is not None and floor_aware_score(lgb_oof, y) >= floor_aware_score(base_oof, y) - 0.005:
            model_pool.append({"name": "lgbm_meta", "oof": lgb_oof, "test": lgb_test})
            evaluate_prob(y, lgb_oof, "LGBM meta")

summarize_pool(model_pool, "Final model pool").to_excel(OUTPUT_DIR / "model_pool_final.xlsx", index=False)


## 13. Ensemble Optimization

In [ ]:
def combine_global(pool, weights):
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.sum()
    oof = np.zeros_like(pool[0]["oof"])
    test = np.zeros_like(pool[0]["test"])
    for w, m in zip(weights, pool):
        oof += w * m["oof"]
        test += w * m["test"]
    return normalize_prob(oof), normalize_prob(test)


def optimize_global_weights(pool, n_trials=4000):
    rng = np.random.default_rng(SEED)
    M = len(pool)
    candidates = []
    candidates.append(np.ones(M) / M)
    scores = np.array([max(floor_aware_score(m["oof"], y), 0.001) for m in pool])
    candidates.append(scores / scores.sum())
    for i in range(M):
        w = np.zeros(M)
        w[i] = 1.0
        candidates.append(w)
    best_w = candidates[0]
    best_oof, best_test = combine_global(pool, best_w)
    best_score = floor_aware_score(best_oof, y)
    for w in candidates:
        po, pt = combine_global(pool, w)
        sc = floor_aware_score(po, y)
        if sc > best_score:
            best_score, best_w, best_oof, best_test = sc, w, po, pt
    alpha_base = np.clip(scores / scores.mean(), 0.25, 3.0)
    for _ in range(n_trials):
        if rng.random() < 0.65:
            w = rng.dirichlet(alpha_base)
        else:
            w = rng.dirichlet(np.ones(M))
        po, pt = combine_global(pool, w)
        sc = floor_aware_score(po, y)
        if sc > best_score:
            best_score, best_w, best_oof, best_test = sc, w, po, pt
    print("Best global floor score:", best_score)
    display(pd.DataFrame({"model": [m["name"] for m in pool], "weight": best_w}))
    return best_oof, best_test, best_w


def optimize_per_class_weights(pool, n_trials=1200):
    rng = np.random.default_rng(SEED + 11)
    M = len(pool)
    oofs = np.stack([m["oof"] for m in pool], axis=0)
    tests = np.stack([m["test"] for m in pool], axis=0)
    base_w = np.ones((NUM_LABELS, M), dtype=np.float64) / M
    best_w = base_w.copy()
    def combine(W):
        po = np.zeros_like(oofs[0])
        pt = np.zeros_like(tests[0])
        for c in range(NUM_LABELS):
            po[:, c] = np.sum(oofs[:, :, c].T * W[c], axis=1)
            pt[:, c] = np.sum(tests[:, :, c].T * W[c], axis=1)
        return normalize_prob(po), normalize_prob(pt)
    best_oof, best_test = combine(best_w)
    best_score = floor_aware_score(best_oof, y)
    model_scores = np.array([max(floor_aware_score(m["oof"], y), 0.001) for m in pool])
    alpha = np.clip(model_scores / model_scores.mean(), 0.25, 3.0)
    for _ in range(n_trials):
        W = np.zeros((NUM_LABELS, M), dtype=np.float64)
        for c in range(NUM_LABELS):
            W[c] = rng.dirichlet(alpha)
        po, pt = combine(W)
        sc = floor_aware_score(po, y)
        if sc > best_score:
            best_score, best_w, best_oof, best_test = sc, W, po, pt
    print("Best per-class floor score:", best_score)
    display(pd.DataFrame(best_w, index=LABELS, columns=[m["name"] for m in pool]))
    return best_oof, best_test, best_w

base_oof, base_test, global_weights = optimize_global_weights(model_pool)
evaluate_prob(y, base_oof, "Global weighted ensemble")
if RUN_PER_CLASS_WEIGHTING and len(model_pool) >= 2:
    pcw_oof, pcw_test, pcw_weights = optimize_per_class_weights(model_pool)
else:
    pcw_oof, pcw_test, pcw_weights = base_oof, base_test, None
evaluate_prob(y, pcw_oof, "Per-class weighted ensemble")


## 14. Calibration, Gates, and Final Tuning

In [ ]:
def apply_prior_correction(prob, tau):
    train_prior = np.bincount(y, minlength=NUM_LABELS).astype(np.float64)
    train_prior = train_prior / train_prior.sum()
    uniform = np.ones(NUM_LABELS, dtype=np.float64) / NUM_LABELS
    mult = (uniform / np.maximum(train_prior, 1e-9)) ** tau
    return normalize_prob(prob * mult.reshape(1, -1))


def tune_prior_tau(prob):
    best_tau = 0.0
    best_p = normalize_prob(prob)
    best_score = floor_aware_score(best_p, y)
    for tau in np.linspace(0.0, 0.45, 16):
        p = apply_prior_correction(prob, tau)
        sc = floor_aware_score(p, y)
        if sc > best_score:
            best_tau, best_score, best_p = float(tau), sc, p
    print("Best prior tau:", best_tau, best_score)
    return best_p, best_tau


def apply_binary_gates(prob, gates, strength=0.12):
    p = normalize_prob(prob).copy()
    if not gates:
        return p
    for lab, gate in gates.items():
        idx = label2id[lab]
        gate = np.asarray(gate)
        q85 = np.quantile(gate, 0.85)
        mask = gate >= q85
        p[mask, idx] = (1 - strength) * p[mask, idx] + strength * gate[mask]
    return normalize_prob(p)


def apply_pairwise(prob, pairs, strength=0.12, margin_threshold=0.16):
    p = normalize_prob(prob).copy()
    if not pairs:
        return p
    top = np.argsort(p, axis=1)[:, -2:]
    top1, top2 = top[:, 1], top[:, 0]
    margin = p[np.arange(len(p)), top1] - p[np.arange(len(p)), top2]
    for (a, b), pp in pairs.items():
        ia, ib = label2id[a], label2id[b]
        mask = (((top1 == ia) & (top2 == ib)) | ((top1 == ib) & (top2 == ia))) & (margin < margin_threshold)
        if mask.any():
            p[mask, ia] = (1 - strength) * p[mask, ia] + strength * pp[mask, 0]
            p[mask, ib] = (1 - strength) * p[mask, ib] + strength * pp[mask, 1]
    return normalize_prob(p)


def keyword_rescue(prob, texts, strength=0.06):
    p = normalize_prob(prob).copy()
    for i, text in enumerate(texts):
        scores = keyword_scores(normalize_text(text))
        if scores[label2id["Ekonomi"]] >= 4 and p[i, label2id["Ekonomi"]] >= 0.08:
            p[i, label2id["Ekonomi"]] += strength
        if scores[label2id["Tata Kelola"]] >= 5 and p[i, label2id["Tata Kelola"]] >= 0.10:
            p[i, label2id["Tata Kelola"]] += strength
        if scores[label2id["Kualitas Pangan"]] >= 5 and p[i, label2id["Kualitas Pangan"]] >= 0.10:
            p[i, label2id["Kualitas Pangan"]] += strength
        if scores[label2id["Politik"]] >= 5 and p[i, label2id["Politik"]] >= 0.10:
            p[i, label2id["Politik"]] += strength * 0.8
    return normalize_prob(p)


def postprocess(prob, gates=None, pairs=None, texts=None):
    p = normalize_prob(prob)
    if RUN_POSTPROCESSING:
        p = apply_binary_gates(p, gates or {})
        p = apply_pairwise(p, pairs or {})
        if texts is not None:
            p = keyword_rescue(p, texts)
    return normalize_prob(p)


def optimize_multipliers(prob, test_prob, n_trials=3000):
    rng = np.random.default_rng(SEED + 99)
    base = normalize_prob(prob)
    test_base = normalize_prob(test_prob)
    best_mult = np.ones(NUM_LABELS, dtype=np.float64)
    best_prob = base.copy()
    best_score = floor_aware_score(best_prob, y)
    ranges = {
        "Anggaran": (0.88, 1.12),
        "Distribusi": (0.92, 1.25),
        "Ekonomi": (0.92, 1.15),
        "Kualitas Pangan": (0.86, 1.10),
        "Lainnya": (0.92, 1.25),
        "Politik": (0.90, 1.35),
        "Sasaran Penerima": (0.92, 1.25),
        "Tata Kelola": (0.92, 1.35),
    }
    for _ in range(n_trials):
        mult = np.ones(NUM_LABELS, dtype=np.float64)
        for lab in LABELS:
            lo, hi = ranges[lab]
            mult[label2id[lab]] = rng.uniform(lo, hi)
        p = normalize_prob(base * mult.reshape(1, -1))
        sc = floor_aware_score(p, y)
        if sc > best_score:
            best_score, best_mult, best_prob = sc, mult, p
    for _ in range(3):
        improved = False
        for lab in LABELS:
            idx = label2id[lab]
            lo, hi = ranges[lab]
            current = best_mult[idx]
            local_best = best_score
            local_val = current
            for val in np.linspace(lo, hi, 20):
                trial = best_mult.copy()
                trial[idx] = val
                p = normalize_prob(base * trial.reshape(1, -1))
                sc = floor_aware_score(p, y)
                if sc > local_best:
                    local_best, local_val = sc, val
            if local_best > best_score:
                best_score = local_best
                best_mult[idx] = local_val
                improved = True
        if not improved:
            break
    final_test = normalize_prob(test_base * best_mult.reshape(1, -1))
    print("Best multiplier score:", best_score)
    display(pd.DataFrame({"label": LABELS, "multiplier": best_mult}))
    return normalize_prob(best_prob), final_test, best_mult

prior_oof, best_tau = tune_prior_tau(pcw_oof)
prior_test = apply_prior_correction(pcw_test, best_tau)
post_oof = postprocess(prior_oof, gate_oof, pair_oof, train_df[TEXT_COL].tolist())
post_test = postprocess(prior_test, gate_test, pair_test, test_df[TEST_TEXT_COL].tolist())
final_oof, final_test, best_mult = optimize_multipliers(post_oof, post_test)
evaluate_prob(y, final_oof, "Final tuned ensemble")
show_confusion(y, final_oof).to_excel(OUTPUT_DIR / "confusion_matrix_final.xlsx")
save_report(y, final_oof, OUTPUT_DIR / "classification_report_final.xlsx")


## 15. Sanity Checks and Submission Files

In [ ]:
def distribution_report(prob_train, prob_test, name):
    train_pred = pred_from_prob(prob_train)
    test_pred = pred_from_prob(prob_test)
    train_true_dist = pd.Series(y).map(id2label).value_counts(normalize=True).reindex(LABELS).fillna(0)
    train_pred_dist = pd.Series(train_pred).map(id2label).value_counts(normalize=True).reindex(LABELS).fillna(0)
    test_pred_dist = pd.Series(test_pred).map(id2label).value_counts(normalize=True).reindex(LABELS).fillna(0)
    df = pd.DataFrame({"train_true": train_true_dist, "train_oof_pred": train_pred_dist, "test_pred": test_pred_dist, "test_minus_train_true": test_pred_dist - train_true_dist})
    df["abs_shift"] = df["test_minus_train_true"].abs()
    display(df)
    df.to_excel(OUTPUT_DIR / f"distribution_{name}.xlsx")
    conf = pd.DataFrame({"set": ["train_oof", "test"], "mean_conf": [normalize_prob(prob_train).max(axis=1).mean(), normalize_prob(prob_test).max(axis=1).mean()], "mean_margin": [margin_prob(prob_train).mean(), margin_prob(prob_test).mean()], "mean_entropy": [entropy_prob(prob_train).mean(), entropy_prob(prob_test).mean()]})
    display(conf)
    conf.to_excel(OUTPUT_DIR / f"confidence_{name}.xlsx", index=False)
    return df, conf


def make_submission(prob, filename):
    pred_labels = [id2label[i] for i in pred_from_prob(prob)]
    sub = template_df.copy()
    if "label" in sub.columns:
        sub["label"] = pred_labels
    else:
        target_cols = [c for c in sub.columns if c != ID_COL]
        if target_cols:
            sub[target_cols[0]] = pred_labels
        else:
            sub["label"] = pred_labels
    if ID_COL in sub.columns and ID_COL in test_df.columns:
        if sub[ID_COL].astype(str).tolist() != test_df[ID_COL].astype(str).tolist():
            raise ValueError("ID submission tidak sama dengan test.")
    path = OUTPUT_DIR / filename
    sub.to_excel(path, index=False)
    print("Saved:", path)
    return sub, path

dist_df, conf_df = distribution_report(final_oof, final_test, "final")

submission_candidates = {
    "base_global": (base_oof, base_test),
    "per_class_weighted": (pcw_oof, pcw_test),
    "prior": (prior_oof, prior_test),
    "postprocessed": (post_oof, post_test),
    "final": (final_oof, final_test),
}

if MAKE_DIVERSE_SUBMISSIONS:
    for lab, mult_value in [("Politik", 1.06), ("Tata Kelola", 1.06), ("Lainnya", 1.05), ("Distribusi", 1.05), ("Sasaran Penerima", 1.05)]:
        m = best_mult.copy()
        m[label2id[lab]] *= mult_value
        submission_candidates[f"final_{lab.lower().replace(' ', '_')}_plus"] = (normalize_prob(post_oof * m.reshape(1, -1)), normalize_prob(post_test * m.reshape(1, -1)))

def candidate_sanity_metrics(po, pt):
    train_true_dist = pd.Series(y).map(id2label).value_counts(normalize=True).reindex(LABELS).fillna(0).values
    train_pred_dist = pd.Series(pred_from_prob(po)).map(id2label).value_counts(normalize=True).reindex(LABELS).fillna(0).values
    test_pred_dist = pd.Series(pred_from_prob(pt)).map(id2label).value_counts(normalize=True).reindex(LABELS).fillna(0).values
    shift_test_train = float(np.mean(np.abs(test_pred_dist - train_true_dist)))
    shift_test_oof = float(np.mean(np.abs(test_pred_dist - train_pred_dist)))
    conf_gap = float(abs(normalize_prob(pt).max(axis=1).mean() - normalize_prob(po).max(axis=1).mean()))
    return shift_test_train, shift_test_oof, conf_gap

rows = []
for name, (po, pt) in submission_candidates.items():
    ba, rec = evaluate_prob(y, po, f"candidate {name}")
    floor_score = floor_aware_score(po, y)
    shift_train, shift_oof, conf_gap = candidate_sanity_metrics(po, pt)
    sanity_score = floor_score - 0.20 * shift_train - 0.10 * shift_oof - 0.05 * conf_gap
    rows.append({"submission": name, "BA": ba, "floor_score": floor_score, "sanity_score": sanity_score, "shift_test_train": shift_train, "shift_test_oof": shift_oof, "conf_gap": conf_gap, **{f"rec_{lab}": rec[label2id[lab]] for lab in LABELS}})
    make_submission(pt, f"{TEAM_NAME}_{name}.xlsx")

summary_df = pd.DataFrame(rows).sort_values(["sanity_score", "floor_score", "BA"], ascending=False).reset_index(drop=True)
display(summary_df)
summary_df.to_excel(OUTPUT_DIR / "submission_candidates_summary.xlsx", index=False)
best_name = summary_df.iloc[0]["submission"]
_, main_submission_path = make_submission(submission_candidates[best_name][1], f"{TEAM_NAME}.xlsx")
try:
    shutil.copy2(main_submission_path, Path(f"{TEAM_NAME}.xlsx"))
    print("Copied main submission to:", Path(f"{TEAM_NAME}.xlsx").resolve())
except Exception as e:
    print("Copy main submission gagal:", repr(e))
print("Recommended submission:", best_name)


## 16. Save Run Summary

In [ ]:
run_summary = {
    "version": "MBG_A100_MAX_CLEAN_V4_TAPT_LOSSDIVERSE",
    "team_name": TEAM_NAME,
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
    "labels": LABELS,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if TRANSFORMERS_AVAILABLE and torch.cuda.is_available() else None,
    "run_tfidf": RUN_TFIDF,
    "run_transformers": RUN_TRANSFORMERS,
    "run_tapt": RUN_TAPT,
    "add_special_keyword_tokens": ADD_SPECIAL_KEYWORD_TOKENS,
    "tfidf_models": list(tfidf_oof_by_model.keys()),
    "transformer_models": list(transformer_oof_by_model.keys()),
    "final_best_multiplier": {lab: float(best_mult[label2id[lab]]) for lab in LABELS},
    "best_prior_tau": float(best_tau),
    "recommended_submission": str(best_name),
    "output_dir": str(OUTPUT_DIR)
}
with open(OUTPUT_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)

readme = f"""
MBG Case 1 A100 MAX CLEAN FIXED V3

Output utama: {TEAM_NAME}.xlsx
Output folder: {OUTPUT_DIR}

Urutan menjalankan notebook:
1. Jalankan semua cell dari atas ke bawah.
2. Upload tiga file xlsx jika diminta.
3. Gunakan file {TEAM_NAME}.xlsx sebagai submission utama.
4. Sertakan notebook ini dan folder output sebagai source code/reproducibility package.

Catatan kepatuhan:
- Tidak menggunakan data eksternal.
- Tidak melakukan anotasi manual data uji.
- Tidak menggunakan generative AI/LLM untuk prediksi, anotasi, pseudo-labeling, atau augmentasi label.
- Model pretrained encoder digunakan untuk supervised fine-tuning klasifikasi.
"""
with open(OUTPUT_DIR / "README_RUN.txt", "w", encoding="utf-8") as f:
    f.write(readme.strip() + "\n")

print(json.dumps(run_summary, indent=2, ensure_ascii=False))
print("Selesai. Ambil output dari:", OUTPUT_DIR)
